In [1]:
import sys
sys.path.append("..")

from src.supertrend import load_raw_data, apply_supertrend
from src.flip_analysis import detect_flips, save_flips

df = load_raw_data("BHARTIARTL")
df = apply_supertrend(df)

trades = detect_flips(df)
trades.head(10)

,direction,status,entry_date,entry_price,initial_sl,initial_sl_pct,exit_date,exit_price,final_sl_at_breach,duration,pct_move,max_adverse_pct,max_gap_against_pct,candles_held
0,UP,CLOSED,2022-01-10 09:15:00+05:30,707.20,682.145500,3.54,2022-01-18 13:15:00+05:30,713.85,703.447060,8 days 04:00:00,0.94,1.77,0.23,14
1,DOWN,CLOSED,2022-01-19 09:15:00+05:30,702.20,739.302907,5.28,2022-01-28 13:15:00+05:30,717.00,725.008963,9 days 04:00:00,-2.11,7.11,0.69,14
2,UP,CLOSED,2022-01-31 09:15:00+05:30,730.95,676.618227,7.43,2022-02-23 13:15:00+05:30,701.20,687.464196,23 days 04:00:00,-4.07,6.29,2.26,36
3,DOWN,CLOSED,2022-02-24 09:15:00+05:30,680.60,716.679400,5.30,2022-03-11 13:15:00+05:30,694.40,699.107070,15 days 04:00:00,-2.03,3.94,1.24,22
4,UP,CLOSED,2022-03-14 09:15:00+05:30,702.85,653.787055,6.98,2022-04-13 13:15:00+05:30,739.90,738.503505,30 days 04:00:00,5.27,2.09,1.11,44
5,DOWN,CLOSED,2022-04-18 09:15:00+05:30,724.95,766.392185,5.72,2022-06-24 13:15:00+05:30,671.00,676.426246,67 days 04:00:00,7.44,5.04,2.18,98
6,UP,CLOSED,2022-06-27 09:15:00+05:30,679.10,641.208692,5.58,2022-07-08 13:15:00+05:30,693.45,669.832156,11 days 04:00:00,2.11,1.93,0.81,20
7,DOWN,CLOSED,2022-07-11 09:15:00+05:30,662.20,703.086060,6.17,2022-07-21 09:15:00+05:30,675.30,675.721780,10 days 00:00:00,-1.98,2.82,1.35,17
8,UP,CLOSED,2022-07-21 13:15:00+05:30,677.20,647.878268,4.33,2022-09-26 09:15:00+05:30,762.65,762.247109,66 days 20:00:00,12.62,1.89,2.26,88
9,DOWN,CLOSED,2022-09-26 13:15:00+05:30,755.00,790.118888,4.65,2022-09-30 09:15:00+05:30,784.00,790.118888,3 days 20:00:00,-3.84,4.06,0.59,8


In [3]:
trades.tail(3)

,direction,status,entry_date,entry_price,initial_sl,initial_sl_pct,exit_date,exit_price,final_sl_at_breach,duration,pct_move,max_adverse_pct,max_gap_against_pct,candles_held
56,UP,CLOSED,2026-06-17 13:15:00+05:30,1874.2,1798.652127,4.03,2026-06-25 09:15:00+05:30,1863.4,1853.833891,7 days 20:00:00,-0.58,1.06,0.44,12
57,DOWN,CLOSED,2026-06-25 13:15:00+05:30,1849.9,1925.126518,4.07,2026-07-02 13:15:00+05:30,1871.2,1903.217480,7 days 00:00:00,-1.15,2.01,0.38,9
58,UP,OPEN,2026-07-03 09:15:00+05:30,1903.6,1824.067497,4.18,NaT,NaN,1865.928900,NaT,NaN,1.50,0.61,12


In [4]:
trades.count()

direction              59
status                 59
entry_date             59
entry_price            59
initial_sl             59
initial_sl_pct         59
exit_date              58
exit_price             58
final_sl_at_breach     59
duration               58
pct_move               58
max_adverse_pct        59
max_gap_against_pct    59
candles_held           59
dtype: int64

In [5]:
save_flips(trades, "BHARTIARTL")

Saved 59 flips to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/BHARTIARTL_flips.csv


'/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/BHARTIARTL_flips.csv'

In [6]:
# Remove timezone info from datetime columns, just for the Excel export
import pandas as pd
trades_excel = trades.copy()

trades_excel["entry_date"] = trades_excel["entry_date"].dt.tz_localize(None)
trades_excel["exit_date"] = pd.to_datetime(trades_excel["exit_date"]).dt.tz_localize(None)

trades_excel.to_excel("../exports/BHARTIARTL_flips.xlsx", index=False, sheet_name="Flips")
print("Saved Excel file to exports/BHARTIARTL_flips.xlsx")

Saved Excel file to exports/BHARTIARTL_flips.xlsx


In [7]:
trades[["max_adverse_pct", "max_gap_against_pct"]].describe()

,max_adverse_pct,max_gap_against_pct
count,59.000000,59.000000
mean,3.005932,1.097966
std,1.977600,0.737537
min,0.770000,-0.000000
25%,1.770000,0.635000
50%,2.680000,0.940000
75%,3.805000,1.385000
max,12.980000,4.250000


In [8]:
# Quick logical check: for CLOSED, losing trades, max_adverse_pct should be >= abs(pct_move)
losing = trades[(trades["status"] == "CLOSED") & (trades["pct_move"] < 0)]
(losing["max_adverse_pct"] >= losing["pct_move"].abs()).all()

np.True_